# 4b Prepare Reviews For Analysis

This notebook prepares all review-side artifacts for downstream analyses.

For each requested condition it builds both the `original` and `rephrased` branches, then saves:
- canonical review master tables
- review panel registry and sampling frame
- proposal-level review score summaries
- review-text embedding bundles
- pairwise cosine matrices
- review-space UMAP caches
- per-proposal review panel distance caches

It keeps the full repeated-review reservoir intact and does not pre-sample exact-n matched AI panels.

In [1]:
CONDITIONS_TO_PREPARE = ['baseline', 'one_at_a_time', 'persona']
TEXT_VERSIONS = ['original', 'rephrased']

EMBEDDING_MODEL_NAME = 'michiyasunaga/BioLinkBERT-large'
REUSE_EXISTING_ARTIFACTS = True
WRITE_JSON_COMPANIONS = True


In [2]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd

import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from prepare_reviews_for_analysis import (
    AI_REVIEW_EXPECTED_PER_MODEL,
    AI_REVIEW_EXPECTED_POOLED,
    AI_REVIEW_EXPECTED_ROWS,
    CANONICAL_SCORE_COLUMNS,
    build_human_target_proposal_lookup,
    build_proposal_review_scores_summary,
    build_review_embedding_bundle,
    build_review_master_table,
    build_review_panel_distance_cache,
    build_review_panel_registry,
    build_review_sampling_frame,
    compute_pairwise_cosine_matrix,
    find_project_root,
    fit_or_load_review_umap,
    load_ai_original_reviews,
    load_ai_rephrased_reviews,
    load_human_original_reviews,
    load_human_rephrased_reviews,
    locate_latest_ai_review_files,
    validate_review_alignment,
    write_review_prepare_manifest,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
human_target_proposal_lookup = build_human_target_proposal_lookup(PROJECT_ROOT)
human_original_reviews_df = load_human_original_reviews(PROJECT_ROOT)
human_rephrased_reviews_df = load_human_rephrased_reviews(PROJECT_ROOT)
human_alignment_issues = validate_review_alignment(human_original_reviews_df, human_rephrased_reviews_df, 'human')
if human_alignment_issues:
    raise RuntimeError('Human review alignment failed: ' + '; '.join(human_alignment_issues))

if human_target_proposal_lookup['target_proposal_uid'].nunique() != 23:
    raise RuntimeError(f'Expected 23 target human proposals, found {human_target_proposal_lookup["target_proposal_uid"].nunique()}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Conditions to prepare: {CONDITIONS_TO_PREPARE}')
print(f'Human target proposals: {len(human_target_proposal_lookup)}')
print(f'Human original reviews: {len(human_original_reviews_df)}')
print(f'Human rephrased reviews: {len(human_rephrased_reviews_df)}')
print('Observed human panel sizes:')
print(human_original_reviews_df.groupby('target_proposal_uid').size().value_counts().sort_index())


Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal
Conditions to prepare: ['baseline', 'one_at_a_time', 'persona']
Human target proposals: 23
Human original reviews: 85
Human rephrased reviews: 85
Observed human panel sizes:
2     1
3     6
4    15
5     1
Name: count, dtype: int64


In [3]:
condition_prepare_outputs = {}

for condition in CONDITIONS_TO_PREPARE:
    print(f'\n=== Prepare reviews: {condition} ===')
    review_files = locate_latest_ai_review_files(PROJECT_ROOT, condition)
    ai_original_reviews_df = load_ai_original_reviews(review_files['original'], condition)
    ai_rephrased_reviews_df = load_ai_rephrased_reviews(review_files['rephrased'], condition)
    ai_alignment_issues = validate_review_alignment(ai_original_reviews_df, ai_rephrased_reviews_df, f'ai::{condition}')
    if ai_alignment_issues:
        raise RuntimeError('AI review alignment failed: ' + '; '.join(ai_alignment_issues))

    if ai_original_reviews_df['review_uid'].nunique() != len(ai_original_reviews_df):
        raise RuntimeError(f'{condition}: duplicate AI review_uid values in original reviews')
    if len(ai_original_reviews_df) != AI_REVIEW_EXPECTED_ROWS:
        print(f'WARNING: {condition} has {len(ai_original_reviews_df)} AI reviews; expected {AI_REVIEW_EXPECTED_ROWS} after redesign.')

    proposal_meta_cols = ['target_proposal_uid', 'target_cohort', 'target_proposal_id', 'target_proposal_title', 'target_proposal_status', 'target_authors', 'target_ranking', 'target_funding']
    ai_original_reviews_df = ai_original_reviews_df.merge(
        human_target_proposal_lookup[proposal_meta_cols],
        on=['target_proposal_uid', 'target_cohort', 'target_proposal_id'],
        how='left',
        validate='many_to_one',
        suffixes=('', '_lookup'),
    )
    ai_rephrased_reviews_df = ai_rephrased_reviews_df.merge(
        human_target_proposal_lookup[proposal_meta_cols],
        on=['target_proposal_uid', 'target_cohort', 'target_proposal_id'],
        how='left',
        validate='many_to_one',
        suffixes=('', '_lookup'),
    )

    branch_outputs = {}
    branch_masters = {}
    for text_version in TEXT_VERSIONS:
        output_dir = PROJECT_ROOT / 'data' / 'prepared' / condition / 'reviews' / text_version
        output_dir.mkdir(parents=True, exist_ok=True)

        master_df = build_review_master_table(
            condition=condition,
            text_version=text_version,
            ai_original_df=ai_original_reviews_df,
            ai_rephrased_df=ai_rephrased_reviews_df,
            human_original_df=human_original_reviews_df,
            human_rephrased_df=human_rephrased_reviews_df,
        )
        branch_masters[text_version] = master_df

        master_csv_path = output_dir / 'review_master.csv'
        master_json_path = output_dir / 'review_master.json'
        embeddings_text_path = output_dir / 'review_embeddings_text.pkl'
        pairwise_text_path = output_dir / 'review_pairwise_cosine_text.npy'
        review_umap_reducer_path = output_dir / 'review_umap_reducer.pkl'
        review_umap_coords_path = output_dir / 'review_umap2d.npy'
        panel_cache_path = output_dir / 'review_panel_distance_cache.pkl'
        manifest_path = output_dir / 'prepare_manifest.json'

        master_df.to_csv(master_csv_path, index=False)
        if WRITE_JSON_COMPANIONS:
            master_json_path.write_text(json.dumps(master_df.to_dict('records'), indent=2, ensure_ascii=False))

        text_bundle = build_review_embedding_bundle(
            master_df,
            text_field='review_text',
            output_path=embeddings_text_path,
            model_name=EMBEDDING_MODEL_NAME,
            pooling='mean',
            reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
        )
        pairwise_text = compute_pairwise_cosine_matrix(text_bundle, pairwise_text_path)
        review_reducer, review_umap2d = fit_or_load_review_umap(
            text_bundle,
            reducer_path=review_umap_reducer_path,
            coords_path=review_umap_coords_path,
            reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
        )
        panel_cache = build_review_panel_distance_cache(master_df, pairwise_text, panel_cache_path)

        strengths_path = output_dir / 'review_embeddings_strengths.pkl'
        weakness_path = output_dir / 'review_embeddings_weakness.pkl'
        pairwise_strengths_path = output_dir / 'review_pairwise_cosine_strengths.npy'
        pairwise_weakness_path = output_dir / 'review_pairwise_cosine_weakness.npy'
        extra_outputs = {}
        if text_version == 'rephrased':
            strengths_bundle = build_review_embedding_bundle(
                master_df,
                text_field='strengths_text',
                output_path=strengths_path,
                model_name=EMBEDDING_MODEL_NAME,
                pooling='mean',
                reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
            )
            weakness_bundle = build_review_embedding_bundle(
                master_df,
                text_field='weakness_text',
                output_path=weakness_path,
                model_name=EMBEDDING_MODEL_NAME,
                pooling='mean',
                reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
            )
            pairwise_strengths = compute_pairwise_cosine_matrix(strengths_bundle, pairwise_strengths_path)
            pairwise_weakness = compute_pairwise_cosine_matrix(weakness_bundle, pairwise_weakness_path)
            extra_outputs = {
                'strengths_embedding_path': strengths_path,
                'weakness_embedding_path': weakness_path,
                'pairwise_strengths_path': pairwise_strengths_path,
                'pairwise_weakness_path': pairwise_weakness_path,
                'pairwise_strengths_shape': list(pairwise_strengths.shape),
                'pairwise_weakness_shape': list(pairwise_weakness.shape),
            }

        manifest = {
            'prepared_at': datetime.now().isoformat(),
            'condition': condition,
            'text_version': text_version,
            'input_original_ai_file': str(review_files['original']),
            'input_rephrased_ai_file': str(review_files['rephrased']),
            'human_original_rows': int(len(human_original_reviews_df)),
            'human_rephrased_rows': int(len(human_rephrased_reviews_df)),
            'ai_original_rows': int(len(ai_original_reviews_df)),
            'ai_rephrased_rows': int(len(ai_rephrased_reviews_df)),
            'review_master_rows': int(len(master_df)),
            'review_uid_order': master_df['review_uid'].astype(str).tolist(),
            'embedding_model_name': EMBEDDING_MODEL_NAME,
            'review_text_embedding_file': str(embeddings_text_path),
            'review_text_pairwise_file': str(pairwise_text_path),
            'review_umap_reducer_file': str(review_umap_reducer_path),
            'review_umap_coords_file': str(review_umap_coords_path),
            'review_panel_distance_cache_file': str(panel_cache_path),
            'pairwise_text_shape': list(pairwise_text.shape),
            'review_umap_shape': list(review_umap2d.shape),
            'panel_cache_targets': int(len(panel_cache['panels'])),
            **extra_outputs,
        }
        write_review_prepare_manifest(manifest_path, manifest)

        branch_outputs[text_version] = {
            'master_df': master_df,
            'master_csv_path': master_csv_path,
            'manifest_path': manifest_path,
            'text_embedding_path': embeddings_text_path,
            'pairwise_text_path': pairwise_text_path,
            'review_umap_path': review_umap_coords_path,
            'panel_cache_path': panel_cache_path,
            **extra_outputs,
        }
        print(f'  {text_version}: saved {len(master_df)} reviews -> {master_csv_path}')

    original_master = branch_masters['original']
    reviews_root = PROJECT_ROOT / 'data' / 'prepared' / condition / 'reviews'
    panel_registry_df = build_review_panel_registry(original_master)
    sampling_frame_df = build_review_sampling_frame(original_master, panel_registry_df)
    proposal_scores_df = build_proposal_review_scores_summary(original_master)
    panel_registry_path = reviews_root / 'review_panel_registry.csv'
    sampling_frame_path = reviews_root / 'review_sampling_frame.csv'
    proposal_scores_path = reviews_root / 'proposal_review_scores_summary.csv'
    panel_registry_df.to_csv(panel_registry_path, index=False)
    sampling_frame_df.to_csv(sampling_frame_path, index=False)
    proposal_scores_df.to_csv(proposal_scores_path, index=False)

    condition_prepare_outputs[condition] = {
        'branches': branch_outputs,
        'panel_registry_df': panel_registry_df,
        'sampling_frame_df': sampling_frame_df,
        'proposal_scores_df': proposal_scores_df,
        'panel_registry_path': panel_registry_path,
        'sampling_frame_path': sampling_frame_path,
        'proposal_scores_path': proposal_scores_path,
    }
    print(f'  panel registry: {panel_registry_path}')
    print(f'  sampling frame: {sampling_frame_path}')
    print(f'  proposal review score summary: {proposal_scores_path}')



=== Prepare reviews: baseline ===
  original: saved 415 reviews -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/baseline/reviews/original/review_master.csv
  rephrased: saved 415 reviews -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/baseline/reviews/rephrased/review_master.csv
  panel registry: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/baseline/reviews/review_panel_registry.csv
  sampling frame: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/baseline/reviews/review_sampling_frame.csv
  proposal review score summary: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/baseline/reviews/proposal_review_scores_summary.csv

=== Prepare reviews: one_at_a_time ===


Embedding texts: 100%|██████████| 52/52 [00:59<00:00,  1.14s/it]
/Users/eveyhuang/Documents/NICO/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  original: saved 416 reviews -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/one_at_a_time/reviews/original/review_master.csv


Embedding texts: 100%|██████████| 52/52 [00:17<00:00,  3.00it/s]
/Users/eveyhuang/Documents/NICO/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Embedding texts: 100%|██████████| 52/52 [00:15<00:00,  3.45it/s]


  rephrased: saved 416 reviews -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/one_at_a_time/reviews/rephrased/review_master.csv
  panel registry: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/one_at_a_time/reviews/review_panel_registry.csv
  sampling frame: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/one_at_a_time/reviews/review_sampling_frame.csv
  proposal review score summary: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/one_at_a_time/reviews/proposal_review_scores_summary.csv

=== Prepare reviews: persona ===


Embedding texts: 100%|██████████| 52/52 [01:04<00:00,  1.25s/it]
/Users/eveyhuang/Documents/NICO/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  original: saved 414 reviews -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/persona/reviews/original/review_master.csv


Embedding texts: 100%|██████████| 52/52 [00:18<00:00,  2.86it/s]
/Users/eveyhuang/Documents/NICO/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Embedding texts: 100%|██████████| 52/52 [00:15<00:00,  3.28it/s]

  rephrased: saved 414 reviews -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/persona/reviews/rephrased/review_master.csv
  panel registry: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/persona/reviews/review_panel_registry.csv
  sampling frame: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/persona/reviews/review_sampling_frame.csv
  proposal review score summary: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/persona/reviews/proposal_review_scores_summary.csv


In [4]:
summary_rows = []
for condition, outputs in condition_prepare_outputs.items():
    for text_version, branch in outputs['branches'].items():
        master_df = branch['master_df']
        summary_rows.append({
            'condition': condition,
            'text_version': text_version,
            'review_rows': len(master_df),
            'human_rows': int((master_df['review_source'] == 'human').sum()),
            'ai_rows': int((master_df['review_source'] == 'ai').sum()),
            'master_csv': str(branch['master_csv_path']),
            'manifest': str(branch['manifest_path']),
            'review_text_embeddings': str(branch['text_embedding_path']),
            'pairwise_text': str(branch['pairwise_text_path']),
            'panel_cache': str(branch['panel_cache_path']),
        })
    summary_rows.append({
        'condition': condition,
        'text_version': 'shared',
        'review_rows': int(len(outputs['sampling_frame_df'])),
        'human_rows': int(outputs['panel_registry_df']['n_human_reviews_available'].sum()),
        'ai_rows': int(outputs['panel_registry_df']['n_ai_reviews_pooled'].sum()),
        'master_csv': str(outputs['panel_registry_path']),
        'manifest': str(outputs['sampling_frame_path']),
        'review_text_embeddings': str(outputs['proposal_scores_path']),
        'pairwise_text': '',
        'panel_cache': '',
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,condition,text_version,review_rows,human_rows,ai_rows,master_csv,manifest,review_text_embeddings,pairwise_text,panel_cache
0,baseline,original,415,85,330,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,baseline,rephrased,415,85,330,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
2,baseline,shared,415,85,330,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,,
3,one_at_a_time,original,416,85,331,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
4,one_at_a_time,rephrased,416,85,331,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
5,one_at_a_time,shared,416,85,331,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,,
6,persona,original,414,85,329,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
7,persona,rephrased,414,85,329,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
8,persona,shared,414,85,329,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,,
